### Agentic RAG With LangGraph

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Set your GEMINI API key
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

# Initialize models
llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [ ]:
# # Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# # Initialize models
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
# embeddings = OpenAIEmbeddings()

In [ ]:
llm

# State Definition

In [ ]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
### Sample Document and VectorStore

sample_texts = [
    "LangGraph is a framework for building agentic applications with LLMs.",
    "RAG (Retrieval-Augmented Generation) is a technique for improving the accuracy of language model outputs by retrieving relevant information from a external knowledge source.",
    "FAISS is a library for efficient similarity search and clustering of dense vectors.",
    "Agentic systems are designed to autonomously perform tasks and make decisions based on their environment and goals."
]

document=[Document(page_content=text) for text in sample_texts]

### Create a vector store from the sample documents
vector_store = FAISS.from_documents(document, embeddings)
retriever = vector_store.as_retriever(k=3)


## Agents Function

In [ ]:
def decide_retrieval(state: AgentState) -> AgentState:
    """
    Decide if we need to retrieve documents based on the question
    """

    question = state["question"]

    # Simple heuristic: if the question contains certain keywords, we decide to retrieve
    retrieval_keywords = ["what", "how", "explain", "describe", "tell me"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)

    return {**state, "needs_retrieval": needs_retrieval}

In [ ]:
def retrieve_documents(state: AgentState) -> AgentState:
    """
    Retrieve relevant documents based on the question
    """
    
    question = state["question"]
    documents = retriever.invoke(question)

    return {**state, "documents": documents}

In [ ]:
def generate_answer(state: AgentState) -> AgentState:
    """
    Generate an answer based on the question and retrieved documents
    """
    
    question = state["question"]
    documents = state.get("documents", [])

    if documents:
        # RAG approach: use doc as context
        context = "\n\n".join(doc.page_content for doc in documents)
        prompt = f"""Based on the following context, answer the question:
        
        Context:
        {context}
        
        Question: {question}
        
        Answer:"""

    else:
        # Direct generation without retrieval
        prompt = f"""Answer the following question: {question}"""

    response = llm.invoke(prompt)
    answer = response.content

    return {**state, "answer": answer}

## Conditional Logic

In [ ]:
def should_retrieve(state: AgentState) -> str:
    """
    Determine the next step based on retrieval decision
    """

    if state["needs_retrieval"]:
        return "retrieve"
    else:
        return "generate"

## Build the Graph

In [ ]:
# Create the state graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("decide", decide_retrieval)
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("generate", generate_answer)

# Set entry point
workflow.set_entry_point("decide")

# Add conditional edges
workflow.add_conditional_edges(
    "decide",
    should_retrieve,
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)

# Add edges
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph
app = workflow.compile()
app

## Test the Agentic System

In [ ]:
def ask_question(question: str):
    """
    Helper function to ask a question to the agent and get an answer
    """
    initial_state = {
        "question": question,
        "documents": [],
        "answer": "",
        "needs_retrieval": False
    }

    result = app.invoke(initial_state)
    return result

In [ ]:
# Test with a question that should trigger retrieval

question1 = "What is Retrieval-Augmented Generation (RAG)?"
result1 = ask_question(question1)
result1

In [ ]:
# Test with another question

question2 = "How does RAG improve the performance of language models?"
result2 = ask_question(question2)

print(f"Question: {question2}")
print(f"Retrieved documents: {len(result2['documents'])}")
print(f"Answer: {result2['answer']}")
print("\n" + "="*50 + "\n")